In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torchaudio 
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, average_precision_score
import matplotlib.pyplot as plt

from BEATs import BEATs, BEATsConfig
device = "cuda" if torch.cuda.is_available() else "cpu"


In [9]:
class AudioDataset(Dataset):
    def __init__(self, root_fol, target_sr=16000):
        self.file_paths = []
        self.labels = []

        self.CLASS_MAP = {
            "Gunshot": 0,
            "Human": 1,
            "Vehicle": 2,
            "Unknown": -1   # special label for OOD
        }

        for class_name, label in self.CLASS_MAP.items():
            class_path = os.path.join(root_fol, class_name)
            if not os.path.isdir(class_path):
                continue

            for file in os.listdir(class_path):
                if file.lower().endswith(".wav"):
                    full_path = os.path.join(class_path, file)
                    try:
                        waveform, sr = torchaudio.load(full_path)
                        if waveform.numel() == 0:
                            continue
                    except:
                        continue
                    self.file_paths.append(full_path)
                    self.labels.append(label)

        self.target_sr = target_sr

    
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):

        MAX_SECONDS = 5 
        MAX_LENGTH = self.target_sr * MAX_SECONDS

        waveform, sr = torchaudio.load(self.file_paths[idx])

        # Convert to mono 
        if waveform.shape[0] > 1:  
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        waveform = waveform.squeeze(0)
        # 🔥 Trim if too long
        if waveform.shape[0] > MAX_LENGTH:
            waveform = waveform[:MAX_LENGTH]

        # 🔥 Pad if too short
        elif waveform.shape[0] < MAX_LENGTH:
            pad_len = MAX_LENGTH - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        label = self.labels[idx]

        return waveform, label


# ===============================
# 3. Collate Function (Padding)
# ===============================
def pad_audio_unlabelled(batch):
    waveforms, labels = zip(*batch)

    lengths = [w.shape[0] for w in waveforms]
    max_len = max(lengths)

    padded = []
    padding_masks = []

    for w in waveforms:
        pad_len = max_len - w.shape[0]
        padded_wave = torch.nn.functional.pad(w, (0, pad_len))
        padded.append(padded_wave)

        mask = torch.zeros(max_len, dtype=torch.bool)
        if pad_len > 0:
            mask[-pad_len:] = True
        padding_masks.append(mask)

    return torch.stack(padded), torch.tensor(labels), torch.stack(padding_masks)


In [10]:
class BEATsClassifier(nn.Module):
    def __init__(self, beats_model, num_classes):
        super().__init__()
        self.beats = beats_model
        self.classifier = nn.Linear(beats_model.cfg.encoder_embed_dim, num_classes)

    def forward(self, x, padding_mask=None):
        features = self.beats.extract_features(x, padding_mask)[0]
        pooled = features.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

In [11]:
checkpoint = torch.load("BEATs_iter3_plus_AS2M.pt", map_location="cpu")
config = BEATsConfig(checkpoint["cfg"])
beats = BEATs(config)
beats.load_state_dict(checkpoint["model"])

model = BEATsClassifier(beats, num_classes=3).to(device)

state_dict = torch.load("beats_finetuned.pth", map_location=device)
model.load_state_dict(state_dict)
model.eval()   # baseline evaluation (no adaptation)


c:\Users\ASHU\anaconda3\envs\masters_env\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


BEATsClassifier(
  (beats): BEATs(
    (post_extract_proj): Linear(in_features=512, out_features=768, bias=True)
    (patch_embedding): Conv2d(1, 512, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (dropout_input): Dropout(p=0.1, inplace=False)
    (encoder): TransformerEncoder(
      (pos_conv): Sequential(
        (0): Conv1d(768, 768, kernel_size=(128,), stride=(1,), padding=(64,), groups=16)
        (1): SamePad()
        (2): GELU(approximate='none')
      )
      (layers): ModuleList(
        (0): TransformerSentenceEncoderLayer(
          (self_attn): MultiheadAttention(
            (dropout_module): Dropout(p=0.1, inplace=False)
            (relative_attention_bias): Embedding(320, 12)
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768

In [12]:
unlabelled_data = AudioDataset(r'C:\Users\ASHU\Videos\Background\full_audio\Testing')

test_loader = DataLoader(dataset=unlabelled_data, batch_size=8, shuffle=False, collate_fn=pad_audio_unlabelled)

In [ ]:
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for audio, labels, padding_mask in test_loader:
        audio, labels, padding_mask = audio.to(device), labels.to(device), padding_mask.to(device)
        logits = model(audio, padding_mask)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        # Save results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Separate known vs unknown
all_probs = np.array(all_probs)
labels_np = np.array(all_labels)
preds_np = np.array(all_preds)

known_mask = labels_np != -1
unknown_mask = labels_np == -1

# Closed-set metrics (known only)
acc = accuracy_score(labels_np[known_mask], preds_np[known_mask])
prec = precision_score(labels_np[known_mask], preds_np[known_mask], average='macro')
rec = recall_score(labels_np[known_mask], preds_np[known_mask], average='macro')
f1 = f1_score(labels_np[known_mask], preds_np[known_mask], average='macro')

print(f"Known Classes -> Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

# Open-set metrics
# Treat unknown as 0, known as 1 for AUROC
y_true = (labels_np != -1).astype(int)
ood_scores = -np.sum(all_probs * np.log(all_probs + 1e-8), axis=1)  # entropy

fpr, tpr, thresholds = roc_curve(y_true, ood_scores)
# Find FPR at closest TPR to 95%
idx = np.argmin(np.abs(tpr - 0.95))
fpr95 = fpr[idx]
print(f"FPR@95%TPR: {fpr95:.4f}")

auroc = roc_auc_score(y_true, ood_scores)
print(f"AUROC (Known vs Unknown): {auroc:.4f}")

# Open-set accuracy: correctly classified known + correctly rejected unknown
correct_known = (labels_np[known_mask] == preds_np[known_mask]).sum()
correct_unknown = (preds_np[unknown_mask] == -1).sum()
open_acc = (correct_known + correct_unknown) / len(labels_np)
print(f"Open-set Accuracy: {open_acc:.4f}")

# Confusion Matrix (include Unknown as class)
labels_for_cm = np.where(labels_np == -1, 3, labels_np)  # map -1 -> 3
preds_for_cm = np.where(preds_np == -1, 3, preds_np)

class_names = ["Gunshot", "Human", "Vehicle", "Unknown"]
cm = confusion_matrix(labels_for_cm, preds_for_cm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", values_format='d')
plt.title("BEATs Baseline Confusion Matrix (Known + Unknown)")
plt.show()

Known Classes -> Accuracy: 0.9944, Precision: 0.9945, Recall: 0.9944, F1: 0.9944


TypeError: can only concatenate list (not "float") to list